|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Reading through the table<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: write the paged attention oracle<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn.functional as F

import cudalib
from tests.helpers import build_paged, rand_kv

Write the reference implementation of paged attention.

This is stage 07, and the point of it is not speed. It is the **oracle**: the
function every kernel you write afterwards gets checked against. Correct on a
shuffled pool, correct on ragged lengths, correct when the unused slots are
full of somebody else's tokens.

Get it right and be slow. Stage 08 is where you get it back.

In [ ]:
### run this cell

torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

S, H, KVH, D, L = 4, 8, 2, 64, 100
BLOCK = 16

K, V = rand_kv(S, KVH, L, D, dev)
q    = torch.randn(S, H, D, device=dev)
kc, vc, block_tables, context_lens = build_paged(K, V, BLOCK)

print(f'pool {tuple(kc.shape)}, block table {tuple(block_tables.shape)}')
print(f'sequence 0 lives in blocks {block_tables[0].tolist()}')

# Exercise 1: the dense oracle first

No paging at all. Attention over the original contiguous tensors, so you have
something to check the paged version against.

In [ ]:
def reference_attention(q, K, V, scale=None):
  """No paging. q (S,H,D), K/V (S,KVH,L,D). This is the oracle."""
  S, H, D = q.shape
  group   = H // K.shape[1]        # query heads per KV head
  scale   = scale or 1.0/math.sqrt(D)
  out = torch.empty_like(q)
  for s in range(S):
    for h in range(H):
      # which KV head does query head h read?
      sc = 
      out[s,h] = 
  return out

want = reference_attention(q, K, V)
print('oracle:', tuple(want.shape))

# Exercise 2: the scatter

Before attention can read the pool, something has to write it. Given a flat
slot per token, put K and V where they belong.

In [ ]:
def write_kv(key_cache, value_cache, key, value, slot_indices):
  """Scatter this step's K/V into the pool.
  key/value (T, KVH, D), slot_indices (T,) flat slots from slot_index().

  A flat slot decomposes as block = slot // block_size, off = slot % block_size.
  This is the 'slot mapping' you will see all over serving code."""
  BS = key_cache.shape[2]
  for i, slot in enumerate(slot_indices.tolist()):
    

kc2 = torch.zeros_like(kc); vc2 = torch.zeros_like(vc)
slots = torch.tensor([block_tables[0,0]*BLOCK + 0, block_tables[0,0]*BLOCK + 1])
write_kv(kc2, vc2, K[0,:, :2].permute(1,0,2), V[0,:, :2].permute(1,0,2), slots)
print('wrote 2 tokens; matches source:',
      torch.allclose(kc2[block_tables[0,0], :, :2], K[0,:, :2]))

# Exercise 3: the gather

Walk the block table, collect the blocks, cut to `context_len`, then do the
same arithmetic as Exercise 1.

In [ ]:
def paged_attention(q, kc, vc, block_tables, context_lens, scale=None):
  S, H, D = q.shape
  KVH, BS = kc.shape[1], kc.shape[2]
  group   = H // KVH
  scale   = scale or 1.0/math.sqrt(D)
  out = torch.empty_like(q)

  for s in range(S):
    n = int(context_lens[s])

    # which physical blocks hold this sequence's first n tokens?
    blocks = 

    # gather them into (KVH, n, D). Careful with the axis order:
    # kc[blocks] is (nblocks, KVH, BS, D) and you want KVH first.
    k = 
    v = 

    for h in range(H):
      kvh = h // group
      sc  = 
      out[s,h] = 
  return out

got = paged_attention(q, kc, vc, block_tables, context_lens)
print('max difference from the oracle:', (got - want).abs().max().item())

# Exercise 4: the two ways this goes wrong quietly

Blocks get recycled, so the slots past `context_len` hold another request's
tokens. And no two sequences are the same length.

In [ ]:
# poison every slot past context_len, the way a recycled block would be
kc3, vc3 = kc.clone(), vc.clone()
for s in range(S):
  for b in range(block_tables.shape[1]):
    phys = int(block_tables[s,b])
    for off in range(BLOCK):
      if b*BLOCK + off >= L:
        

after = paged_attention(q, kc3, vc3, block_tables, context_lens)
print('output unchanged:', torch.allclose(got, after, atol=1e-5))

# and every sequence may be a different length
ragged = torch.tensor([100, 1, 17, 64], dtype=torch.int32, device=dev)
r = paged_attention(q, kc, vc, block_tables, ragged)
print('ragged context lengths ran:', tuple(r.shape))

# Exercise 5: what did paging cost?

In [ ]:
if dev == 'cuda':
  S, H, KVH, D, L = 32, 16, 8, 128, 512
  K2, V2 = rand_kv(S, KVH, L, D, dev, dtype=torch.float16)
  q2 = torch.randn(S, H, D, device=dev, dtype=torch.float16)
  kcb, vcb, btb, ctxb = build_paged(K2, V2, BLOCK)

  paged = 
  dense = 
  print(f'contiguous SDPA: {dense:8.3f} ms')
  print(f'your paged loop: {paged:8.3f} ms   ({paged/dense:.0f}x slower)')

### Before you open the solution

1. Your paged version is much slower than SDPA. Name the two separate
   reasons, both of which are lines you wrote.
2. `kc[blocks]` produces a new tensor. How big is it, for 32 sequences
   of 512 tokens? Compare that with the size of the output you wanted.
3. Exercise 4 poisoned the slots past `context_len` and your output did
   not move. Which line of your gather is responsible, and what happens
   if you drop it?